# Task 3: Preprocessing Pipeline
Purpose: Build, fit, and save a pipeline that transforms raw data into a model-ready format.
- Fits ONLY on the training split to prevent data leakage.
- Handles log-transform for area, standard scaling for numerics, and one-hot encoding for categoricals.
- Drops one category per feature to prevent the Dummy Variable Trap for Linear/Ridge models.

In [5]:
# load data, feat eng, and split
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_PATH = Path("../data/processed/merged_rice_yield_dataset.csv")
ARTIFACTS_DIR = Path("../artifacts")

df = pd.read_csv(DATA_PATH)

df["temp_mean_c"] = (df["temp_max_c"] + df["temp_min_c"]) / 2
df["temp_range_c"] = df["temp_max_c"] - df["temp_min_c"]

with open(ARTIFACTS_DIR / "split_provinces.json", "r") as f:
    split_info = json.load(f)

train_provinces = split_info["train_provinces"]
test_provinces = split_info["test_provinces"]

train_mask = df["province"].isin(train_provinces)
test_mask = df["province"].isin(test_provinces)

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

X_train = df_train.drop(columns=["yield_mt_ha", "province"])
y_train = df_train["yield_mt_ha"]

X_test = df_test.drop(columns=["yield_mt_ha", "province"])
y_test = df_test["yield_mt_ha"]

print(f"Train features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")

Train features shape: (229, 21)
Test features shape: (58, 21)


In [6]:
# define and fit preprocessing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.pipeline import Pipeline

cat_cols = ["season_period", "ecosystem", "island_group"]
num_cols_std = [
    "n_application_rate", "p_application_rate", "k_application_rate",
    "total_fertilizer_kg_ha", "fertilizer_diversification_index",
    "rainfall_mm", "solar_radiation_mj_m2", "soil_ph", 
    "temp_mean_c", "temp_range_c"
]
num_cols_log = ["area_harvested_ha"]

std_pipe = Pipeline(steps=[("scaler", StandardScaler())])
log_pipe = Pipeline(steps=[
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num_std", std_pipe, num_cols_std),
        ("num_log", log_pipe, num_cols_log),
        ("cat", cat_pipe, cat_cols)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed Train shape: {X_train_processed.shape}")
print(f"Processed Test shape: {X_test_processed.shape}")

Processed Train shape: (229, 15)
Processed Test shape: (58, 15)


In [7]:
# extract feature and save pipeline
import joblib

feature_names = preprocessor.get_feature_names_out()
print(f"Total engineered features: {len(feature_names)}")
print("First 5 feature names:", feature_names[:5])

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(preprocessor, MODELS_DIR / "preprocessing_pipeline.joblib")

with open(ARTIFACTS_DIR / "feature_order.json", "w") as f:
    json.dump(list(feature_names), f, indent=2)

print("\nPipeline and feature order successfully saved!")

Total engineered features: 15
First 5 feature names: ['num_std__n_application_rate' 'num_std__p_application_rate'
 'num_std__k_application_rate' 'num_std__total_fertilizer_kg_ha'
 'num_std__fertilizer_diversification_index']

Pipeline and feature order successfully saved!
